# Simple RAG with Hugging Face - "Attention is All You Need" Paper

This notebook implements a Retrieval-Augmented Generation (RAG) system using:
- Hugging Face API for LLM and embeddings
- The "Attention is All You Need" paper as the knowledge base
- FAISS for vector storage and retrieval

In [39]:
# Install required packages
import subprocess
subprocess.run(["pip", "install", "-q", "langchain", "langchain-community", "langchain-huggingface", "faiss-cpu", "pypdf"], check=True)

CompletedProcess(args=['pip', 'install', '-q', 'langchain', 'langchain-community', 'langchain-huggingface', 'faiss-cpu', 'pypdf'], returncode=0)

In [3]:
# Import libraries and set up API key
import os
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
hf_api_key = os.getenv('HF_API_KEY')

if not hf_api_key:
    raise ValueError("Hugging Face API key not found! Please set HF_API_KEY in your .env file")

print("Hugng Face API key loaded successfully")
os.environ["HF_TOKEN"] = hf_api_key

Hugng Face API key loaded successfully


In [4]:
# Load the "Attention is All You Need" PDF
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "attention is all you need.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages from the PDF")
print(f"\nFirst page preview (first 500 characters):")
print(documents[0].page_content[:500])

Loaded 15 pages from the PDF

First page preview (first 500 characters):
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


In [5]:
from langchain_community. embeddings import OllamaEmbeddings
from langchain_community. vectorstores import FAISS
db=FAISS.from_documents(documents[ : 20] ,OllamaEmbeddings())

ValueError: Error raised by inference endpoint: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/embeddings (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000026C73674B30>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [ ]:
# Create embeddings for all chunks and store in FAISS
from langchain_community.vectorstores import FAISS

# First, create embeddings for all documents
print("Creating embeddings for document chunks...")
texts = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]

# Create FAISS vector store
vector_store = FAISS.from_texts(
    texts=texts,
    embedding=embeddings,
    metadatas=metadatas
)

print(f"Vector store created with {len(chunks)} documents")

# Save the vector store for future use
vector_store.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

In [ ]:
# Set up the LLM using Hugging Face Inference API
from langchain_huggingface import HuggingFaceEndpoint

# Using a smaller model that works well for question answering
llm = HuggingFaceEndpoint(
    repo_id="microsoft/Phi-3-mini-4k-instruct",
    token=hf_api_key,
    temperature=0.7,
    max_new_tokens=500
)

print("LLM loaded successfully (microsoft/Phi-3-mini-4k-instruct)")

In [ ]:
# Create the RAG chain
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Create a custom prompt for question answering
prompt_template = """You are a helpful assistant answering questions about the "Attention is All You Need" paper.
Use the following context from the paper to answer the question. If you don't know the answer from the context, say so.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Create the retrieval QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

print("RAG chain created successfully")

In [ ]:
# Test the RAG system with sample questions
questions = [
    "What is the main contribution of the Attention is All You Need paper?",
    "What is self-attention?",
    "What are the key components of the Transformer architecture?",
    "How does multi-head attention work?"
]

print("=" * 60)
print("Testing RAG System with Sample Questions")
print("=" * 60)

for i, question in enumerate(questions, 1):
    print(f"\nQuestion {i}: {question}")
    print("-" * 50)
    
    result = qa_chain.invoke({"query": question})
    print(f"Answer: {result['result']}")
    print()

In [ ]:
# Interactive question answering function
def ask_question(question):
    """Ask a question about the paper and get an answer."""
    result = qa_chain.invoke({"query": question})
    return result['result']

# Example usage
print("Interactive Question Answering")
print("Type your questions about the 'Attention is All You Need' paper:")
print("(Press Enter after each question, type 'quit' to exit)\n")

# Example question
response = ask_question("What is the difference between encoder and decoder?")
print(f"Question: What is the difference between encoder and decoder?")
print(f"Answer: {response}")

## Summary

This notebook demonstrates a simple RAG (Retrieval-Augmented Generation) system that:

1. **Loads the PDF**: Reads the "Attention is All You Need" paper
2. **Splits the document**: Creates manageable chunks for processing
3. **Creates embeddings**: Uses Hugging Face Inference API for vector representation (no local sentence-transformers needed!)
4. **Stores in FAISS**: Efficient vector storage for similarity search
5. **Uses Hugging Face LLM**: Phi-3-mini model for question answering
6. **Retrieval + Generation**: Combines retrieval from the document with LLM generation

You can now ask questions about the paper using the `ask_question()` function!

In [2]:
from langchain.embeddings import OllamaEmbeddings